In [ ]:
#Data Aggregation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import os
from docx import Document
from docx.shared import Pt, Inches

df_main = pd.read_csv(os.path.join('data', 'cleanedSelfEmployed.csv'))
df_controls = pd.read_csv(os.path.join('data', 'cleanedLabourStats.csv'))

main_dataset = os.path.join('data', 'aggregatedData.csv')

for df in [df_main, df_controls]:
    if not df.empty:
        df['Province'] = df['Province'].astype(str).str.strip()
        df['Sex'] = df['Sex'].astype(str).str.strip()
        df['Year'] = df['Year'].astype(int)

df = pd.merge(df_main, df_controls, on=['Province', 'Sex', 'Year'], how='left')

df['Self_Employment_Rate'] = df['Self_Employed'] / df['Control_LaborForce']

df['log_Rate'] = np.log(df['Self_Employment_Rate'])

df['Treat'] = (df['Sex'] == 'Women+').astype(int)

df['Post'] = (df['Year'] >= 2018).astype(int)

df['Treat_x_Post'] = df['Treat'] * df['Post']

df['Time_Trend'] = df['Year'] - 2014


final_cols = ['Province', 'Year', 'Sex', 'log_Rate', 'Treat', 'Post', 'Treat_x_Post', 'Control_UnemploymentRate', 'Time_Trend']

df_final = df[final_cols].dropna()

df_final.to_csv(main_dataset, index=False)

In [ ]:
#Parallel Trends

sns.set_theme(style="whitegrid")

df_pre = df_final[df_final['Year'] < 2018].copy()

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_final, x='Year', y='log_Rate', hue='Sex', marker='o', linewidth=2.5)

plt.axvline(x=2018, color='red', linestyle='--', label='Einführung WES (2018)')
plt.title('Parallel Trends Check: Entwicklung der Gründungsneigung (log Rate)', fontsize=14)
plt.ylabel('Log(Self Employed / Labor Force)')
plt.legend()
plt.savefig('parallelTrends.png', dpi=300, bbox_inches='tight')
plt.show()

model_pre = smf.ols("log_Rate ~ Treat * C(Year) + C(Province)", data=df_pre)
res_pre = model_pre.fit(cov_type='cluster', cov_kwds={'groups': df_pre['Province']})

print(res_pre.summary())

interaction_vars = [var for var in res_pre.params.index if 'Treat:C(Year)' in var]
f_test = res_pre.f_test(interaction_vars)

print(f_test.summary())

doc = Document()
section = doc.sections[0]
section.page_width = Inches(11)
section.page_height = Inches(8.5)
doc.add_heading('OLS Parallel Trends Regression Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(res_pre.summary()))
run.font.name = 'Courier New'  
run.font.size = Pt(9)  
doc.save("Parallel_Trends_OLS.docx")

doc = Document()    
doc.add_heading('F Test Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(f_test.summary()))
run.font.name = 'Courier New'  
run.font.size = Pt(9)  
doc.save("F_Test_Results.docx")

In [ ]:
#Regression 
formula = """
log_Rate ~ Treat + Treat:Post + 
           Control_UnemploymentRate + 
           C(Province) + C(Year)
"""

model = smf.ols(formula=formula, data=df_final)
results = model.fit(cov_type='cluster', cov_kwds={'groups': df_final['Province']})

print(results.summary())

doc = Document()
section = doc.sections[0]
section.page_width = Inches(11)
section.page_height = Inches(8.5)
doc.add_heading('OLS DiD Regression Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(results.summary()))
run.font.name = 'Courier New'  
run.font.size = Pt(9)  
doc.save("DiD_Results.docx")